# LeetCode 541 — Reverse String II bằng Python

**Link đề:** https://leetcode.com/problems/reverse-string-ii/
**Độ khó:** Easy · **Chủ đề:** Two Pointers, String

```python
class Solution:
    def reverseStr(self, s: str, k: int) -> str:
        ...
```

## Mục tiêu học
1. Đọc hiểu đúng quy tắc "cứ mỗi `2k` ký tự, đảo `k` ký tự đầu" — kể cả hai trường hợp phần cuối bị thiếu.
2. Nắm chắc **slicing** `s[a:b]`, `s[::-1]`, `range(start, stop, step)` và việc slicing **không báo lỗi khi vượt biên**.
3. Hiểu vì sao chuỗi Python **bất biến** (immutable) và cách làm việc qua `list` + `"".join(...)`.
4. Viết 4 lời giải: mô phỏng theo khối → slicing gọn → hai con trỏ trên list → một biểu thức theo chỉ số.
5. Trace từng khối, chứng minh đúng, phân tích Time/Space, tránh các lỗi off-by-one hay gặp.

## 1. Đề bài

Cho chuỗi `s` và số nguyên `k`. Tính từ đầu chuỗi, **cứ mỗi `2k` ký tự** thì **đảo ngược `k` ký tự đầu tiên** của nhóm đó.

- Nếu phần còn lại **ít hơn `k`** ký tự → đảo **toàn bộ** phần còn lại.
- Nếu phần còn lại **từ `k` đến dưới `2k`** ký tự → đảo `k` ký tự đầu, giữ nguyên phần sau.

| `s` | `k` | Output | Giải thích ngắn |
|---|---:|---|---|
| `"abcdefg"` | 2 | `"bacdfeg"` | khối `abcd` → `ba`+`cd`; khối `efg` → `fe`+`g` |
| `"abcd"` | 2 | `"bacd"` | một khối `abcd` → `ba`+`cd` |
| `"abc"` | 5 | `"cba"` | còn 3 < k=5 → đảo hết |
| `"a"` | 1 | `"a"` | |

**Constraints:** `1 <= s.length <= 10^4`, `s` chỉ gồm chữ thường, `1 <= k <= 10^4`.

Nhận xét đầu tiên: `k` có thể **lớn hơn** độ dài chuỗi — code phải chịu được.

## 2. Nền tảng Python cần dùng

### 2.1. Indexing và slicing chuỗi

- `s[i]` lấy ký tự ở vị trí `i` (đếm từ 0).
- `s[a:b]` lấy đoạn từ `a` đến **trước** `b` (không lấy `b`).
- `s[::-1]` đảo ngược cả chuỗi; `s[a:b][::-1]` đảo một đoạn.

Điểm cực kỳ quan trọng cho bài này: **slicing không báo lỗi khi `b` vượt quá độ dài** — Python tự cắt đến hết chuỗi. Trong khi `s[i]` với `i` vượt biên sẽ báo `IndexError`.

In [1]:
s = "abcdefg"
print(s[0], s[3])          # a d
print(s[0:2])              # ab  (vị trí 0,1)
print(s[2:4])              # cd
print(s[0:2][::-1])        # ba  (đảo đoạn 0..1)
print(s[4:100])            # efg (100 vượt biên nhưng không lỗi)
print(repr(s[10:12]))      # ''  (bắt đầu đã vượt biên -> chuỗi rỗng)

a d
ab
cd
ba
efg
''


In [2]:
try:
    print(s[100])
except IndexError as e:
    print("s[100] báo lỗi:", e)

s[100] báo lỗi: string index out of range


### 2.2. `range(start, stop, step)` — nhảy theo bước

Ta cần đi tới đầu mỗi khối `2k`: vị trí `0, 2k, 4k, ...`. Đó chính là `range(0, len(s), 2 * k)`.

In [3]:
s, k = "abcdefg", 2
print(list(range(0, len(s), 2 * k)))   # [0, 4] -> hai khối bắt đầu ở 0 và 4
print(list(range(0, 10, 3)))           # [0, 3, 6, 9]

[0, 4]
[0, 3, 6, 9]


### 2.3. Chuỗi bất biến, `list(s)` và `"".join`

Chuỗi Python **không sửa được tại chỗ**: `s[0] = "z"` báo `TypeError`. Muốn sửa từng ký tự, ta đổi sang list, sửa, rồi ghép lại bằng `"".join(list)`.

In [4]:
s = "abc"
try:
    s[0] = "z"
except TypeError as e:
    print("Lỗi:", e)

chars = list(s)        # ['a', 'b', 'c']
chars[0] = "z"
print(chars, "->", "".join(chars))

Lỗi: 'str' object does not support item assignment
['z', 'b', 'c'] -> zbc


### 2.4. Đổi chỗ hai phần tử `a, b = b, a`

Python tính vế phải thành tuple trước rồi mới gán, nên không cần biến tạm. Đây là nền tảng của kỹ thuật **hai con trỏ** đảo đoạn.

In [5]:
chars = list("abcd")
left, right = 0, 3
while left < right:
    chars[left], chars[right] = chars[right], chars[left]
    left += 1
    right -= 1
print("".join(chars))   # dcba

dcba


## 3. Hiểu đề bằng cách chia khối bằng tay

Với `s = "abcdefg"`, `k = 2` → kích thước khối `2k = 4`:

```text
vị trí :  0 1 2 3 | 4 5 6
ký tự  :  a b c d | e f g
khối   :  [ khối 0 ] [khối 1]
```

| Khối | Bắt đầu `i` | Phần đảo `s[i:i+k]` | Phần giữ `s[i+k:i+2k]` | Kết quả khối |
|---|---:|---|---|---|
| 0 | 0 | `ab` → `ba` | `cd` | `bacd` |
| 1 | 4 | `ef` → `fe` | `g` | `feg` |

Ghép: `"bacd" + "feg" = "bacdfeg"` ✔.

Quan sát: **hai trường hợp đặc biệt của đề tự động được xử lý** nếu ta luôn đảo `s[i:i+k]`:
- còn < k ký tự: `s[i:i+k]` chỉ lấy được phần còn lại → đảo hết ✔
- còn từ k đến < 2k: đảo đúng k ký tự đầu, phần sau giữ nguyên ✔

Nhờ slicing "không lỗi khi vượt biên", ta không cần `if` riêng cho từng trường hợp.

In [6]:
s, k = "abcdefg", 2
for i in range(0, len(s), 2 * k):
    block = s[i:i + 2 * k]
    print(f"i={i}: khối={block!r}, đảo={s[i:i+k]!r} -> {s[i:i+k][::-1]!r}, giữ={s[i+k:i+2*k]!r}")

i=0: khối='abcd', đảo='ab' -> 'ba', giữ='cd'
i=4: khối='efg', đảo='ef' -> 'fe', giữ='g'


## 4. Cách 1 — Mô phỏng theo khối, viết `if` đầy đủ (baseline)

Người mới thường dịch đề sát từng chữ: tính số ký tự còn lại `remain`, rồi xét 3 trường hợp. Cách này dài nhưng giúp ta **chắc chắn hiểu đề**.

In [7]:
def reverse_str_explicit(s: str, k: int) -> str:
    result = []
    i = 0
    n = len(s)
    while i < n:
        remain = n - i
        if remain < k:                       # ít hơn k: đảo hết
            result.append(s[i:][::-1])
        elif remain < 2 * k:                 # từ k đến < 2k: đảo k đầu, giữ phần sau
            result.append(s[i:i + k][::-1])
            result.append(s[i + k:])
        else:                                # đủ 2k: đảo k đầu, giữ k sau
            result.append(s[i:i + k][::-1])
            result.append(s[i + k:i + 2 * k])
        i += 2 * k
    return "".join(result)

print(reverse_str_explicit("abcdefg", 2))  # bacdfeg
print(reverse_str_explicit("abcd", 2))     # bacd
print(reverse_str_explicit("abc", 5))      # cba

bacdfeg
bacd
cba


### Nút thắt của cách 1

Code đúng nhưng có **3 nhánh gần giống nhau**. Nhìn kỹ: cả 3 nhánh đều là "đảo `s[i:i+k]`, giữ `s[i+k:i+2k]`" — chỉ khác ở chỗ đoạn có bị cắt cụt hay không. Mà slicing tự cắt cụt! ⇒ gộp được thành **một** dòng.

Đây là insight chính của bài:

> Với mỗi `i` trong `range(0, n, 2k)`: kết quả khối = `s[i:i+k][::-1] + s[i+k:i+2k]`.

In [8]:
s = "abc"; k = 5; i = 0
print(repr(s[i:i+k][::-1]), "+", repr(s[i+k:i+2*k]))   # 'cba' + ''
s = "abcde"; k = 2; i = 4
print(repr(s[i:i+k][::-1]), "+", repr(s[i+k:i+2*k]))   # 'e' + ''

'cba' + ''
'e' + ''


## 5. Cách 2 — Slicing gọn (lời giải chính)

Ghép từng mảnh đã nói ở trên:
1. Duyệt `i` qua đầu các khối: `range(0, len(s), 2 * k)`.
2. Với mỗi khối, nối `s[i:i+k][::-1]` (phần đảo) và `s[i+k:i+2*k]` (phần giữ).
3. Gom vào list `parts`, cuối cùng `"".join(parts)`.

Vì sao gom vào list rồi `join` thay vì `result += ...`? Nối chuỗi liên tục có thể tạo nhiều chuỗi trung gian; `join` ghép một lần, là thói quen tốt (CPython thường tối ưu `+=` nhưng không đảm bảo).

In [9]:
def reverse_str_slice(s: str, k: int) -> str:
    parts = []
    for i in range(0, len(s), 2 * k):
        parts.append(s[i:i + k][::-1])      # k ký tự đầu của khối, đảo
        parts.append(s[i + k:i + 2 * k])    # k ký tự sau, giữ nguyên
    return "".join(parts)

for s, k in [("abcdefg", 2), ("abcd", 2), ("abc", 5), ("a", 1), ("abcdefgh", 3)]:
    print(f"{s!r:>12}, k={k} -> {reverse_str_slice(s, k)!r}")

   'abcdefg', k=2 -> 'bacdfeg'
      'abcd', k=2 -> 'bacd'
       'abc', k=5 -> 'cba'
         'a', k=1 -> 'a'
  'abcdefgh', k=3 -> 'cbadefhg'


## 6. Trace từng bước

Ví dụ khó hơn một chút: `s = "abcdefghij"`, `k = 3` → khối `2k = 6`.

| Bước | `i` | `s[i:i+3]` | đảo | `s[i+3:i+6]` | parts sau bước |
|---:|---:|---|---|---|---|
| 1 | 0 | `abc` | `cba` | `def` | `['cba','def']` |
| 2 | 6 | `ghi` | `ihg` | `j` | `['cba','def','ihg','j']` |

Kết quả: `"cbadefihgj"`. Ở bước 2 còn 4 ký tự (k ≤ 4 < 2k) → đảo 3 ký tự đầu, giữ `j`.

In [10]:
def trace_reverse_str(s: str, k: int) -> str:
    parts = []
    print(f"s={s!r}, k={k}, các điểm bắt đầu khối = {list(range(0, len(s), 2 * k))}")
    for step, i in enumerate(range(0, len(s), 2 * k), start=1):
        head = s[i:i + k]
        tail = s[i + k:i + 2 * k]
        parts.append(head[::-1])
        parts.append(tail)
        print(f"Bước {step}: i={i:>2} | đảo {head!r:>6} -> {head[::-1]!r:>6} | giữ {tail!r:>6} | parts={parts}")
    ans = "".join(parts)
    print("Kết quả:", repr(ans))
    return ans

trace_reverse_str("abcdefghij", 3)
print()
trace_reverse_str("abcdefg", 2)

s='abcdefghij', k=3, các điểm bắt đầu khối = [0, 6]
Bước 1: i= 0 | đảo  'abc' ->  'cba' | giữ  'def' | parts=['cba', 'def']
Bước 2: i= 6 | đảo  'ghi' ->  'ihg' | giữ    'j' | parts=['cba', 'def', 'ihg', 'j']
Kết quả: 'cbadefihgj'

s='abcdefg', k=2, các điểm bắt đầu khối = [0, 4]
Bước 1: i= 0 | đảo   'ab' ->   'ba' | giữ   'cd' | parts=['ba', 'cd']
Bước 2: i= 4 | đảo   'ef' ->   'fe' | giữ    'g' | parts=['ba', 'cd', 'fe', 'g']
Kết quả: 'bacdfeg'


'bacdfeg'

Giải thích `enumerate(iterable, start=1)`: trả về từng cặp `(số thứ tự, phần tử)`; `start=1` để đánh số từ 1 cho dễ đọc.

## 7. Code nộp LeetCode — Cách slicing

In [11]:
class Solution:
    def reverseStr(self, s: str, k: int) -> str:
        parts = []
        for i in range(0, len(s), 2 * k):
            parts.append(s[i:i + k][::-1])
            parts.append(s[i + k:i + 2 * k])
        return "".join(parts)

sol = Solution()
print(sol.reverseStr("abcdefg", 2))  # bacdfeg
print(sol.reverseStr("abcd", 2))     # bacd

bacdfeg
bacd


## 8. Cách 3 — Hai con trỏ trên list (đảo tại chỗ)

Đây là cách "kinh điển" trong ngôn ngữ như C++/Java: đổi chuỗi thành mảng ký tự, rồi với mỗi khối đảo đoạn `[i, j]` bằng hai con trỏ.

Điểm phải cẩn thận: đầu phải `j = min(i + k - 1, n - 1)` — dùng `min` để không vượt cuối mảng (truy cập phần tử thì **có** lỗi vượt biên, khác slicing).

In [12]:
def reverse_str_two_pointers(s: str, k: int) -> str:
    chars = list(s)
    n = len(chars)
    for i in range(0, n, 2 * k):
        left = i
        right = min(i + k - 1, n - 1)
        while left < right:
            chars[left], chars[right] = chars[right], chars[left]
            left += 1
            right -= 1
    return "".join(chars)

print(reverse_str_two_pointers("abcdefg", 2))
print(reverse_str_two_pointers("abc", 5))

bacdfeg
cba


In [13]:
def trace_two_pointers(s: str, k: int) -> None:
    chars = list(s)
    n = len(chars)
    for i in range(0, n, 2 * k):
        left, right = i, min(i + k - 1, n - 1)
        print(f"Khối i={i}: đảo đoạn [{left}, {right}]")
        while left < right:
            chars[left], chars[right] = chars[right], chars[left]
            print(f"   swap({left},{right}) -> {''.join(chars)}")
            left += 1
            right -= 1
    print("Kết quả:", "".join(chars))

trace_two_pointers("abcdefg", 3)

Khối i=0: đảo đoạn [0, 2]
   swap(0,2) -> cbadefg
Khối i=6: đảo đoạn [6, 6]
Kết quả: cbadefg


### Code nộp LeetCode — Hai con trỏ

In [14]:
class Solution:
    def reverseStr(self, s: str, k: int) -> str:
        chars = list(s)
        n = len(chars)
        for i in range(0, n, 2 * k):
            left, right = i, min(i + k - 1, n - 1)
            while left < right:
                chars[left], chars[right] = chars[right], chars[left]
                left += 1
                right -= 1
        return "".join(chars)

print(Solution().reverseStr("abcdefg", 2))

bacdfeg


### Biến thể gọn: gán slice cho list

List (khác chuỗi) cho phép **gán vào một slice**: `chars[i:i+k] = reversed(chars[i:i+k])`. Python tự thay đoạn đó, kể cả khi đoạn ngắn hơn `k` ở cuối.

In [15]:
def reverse_str_slice_assign(s: str, k: int) -> str:
    chars = list(s)
    for i in range(0, len(chars), 2 * k):
        chars[i:i + k] = reversed(chars[i:i + k])
    return "".join(chars)

print(reverse_str_slice_assign("abcdefg", 2))
print(reverse_str_slice_assign("abcdefghij", 3))

bacdfeg
cbadefihgj


## 9. Cách 4 — Tính trực tiếp ký tự ở mỗi vị trí (sáng tạo)

Thay vì nghĩ theo khối, hỏi: **ký tự ở vị trí `j` của kết quả đến từ đâu?**

- Khối chứa `j` bắt đầu tại `start = j - j % (2k)`.
- Nếu `j - start >= k` → `j` ở nửa "giữ nguyên" → lấy `s[j]`.
- Ngược lại `j` ở nửa "đảo". Đoạn đảo là `[start, end]` với `end = min(start + k, n) - 1`. Đảo đoạn thì vị trí `j` nhận ký tự ở vị trí đối xứng `start + end - j`.

Công thức đối xứng: trong đoạn `[start, end]`, phần tử cách đầu `d` bước sẽ sang vị trí cách cuối `d` bước: `end - (j - start) = start + end - j`.

In [16]:
def reverse_str_index(s: str, k: int) -> str:
    n = len(s)
    out = []
    for j in range(n):
        start = j - j % (2 * k)
        if j - start >= k:
            out.append(s[j])
        else:
            end = min(start + k, n) - 1
            out.append(s[start + end - j])
    return "".join(out)

print(reverse_str_index("abcdefg", 2))
print(reverse_str_index("abc", 5))

bacdfeg
cba


In [17]:
# Kiểm tra công thức đối xứng trên một ví dụ nhỏ
s, k = "abcdefg", 3
for j in range(len(s)):
    start = j - j % (2 * k)
    if j - start >= k:
        print(f"j={j}: nửa giữ -> s[{j}]={s[j]}")
    else:
        end = min(start + k, len(s)) - 1
        print(f"j={j}: nửa đảo đoạn [{start},{end}] -> s[{start + end - j}]={s[start + end - j]}")

j=0: nửa đảo đoạn [0,2] -> s[2]=c
j=1: nửa đảo đoạn [0,2] -> s[1]=b
j=2: nửa đảo đoạn [0,2] -> s[0]=a
j=3: nửa giữ -> s[3]=d
j=4: nửa giữ -> s[4]=e
j=5: nửa giữ -> s[5]=f
j=6: nửa đảo đoạn [6,6] -> s[6]=g


Cách này ít dùng khi nộp bài, nhưng luyện kỹ năng **ánh xạ chỉ số** (index mapping) — hữu ích cho nhiều bài khác như xoay mảng, reshape ma trận.

## 10. Vì sao đúng?

**Invariant** (bất biến) của cách 2: sau khi xử lý xong các khối bắt đầu tại `0, 2k, ..., i`, thì `"".join(parts)` đúng bằng kết quả mong muốn của tiền tố `s[:i+2k]` (hoặc cả chuỗi nếu khối cuối).

- Các khối **không chồng nhau** và phủ kín `s` (vì bước nhảy đúng `2k`, mỗi khối lấy `s[i:i+k]` và `s[i+k:i+2k]` — liền nhau, tổng là `s[i:i+2k]`).
- Trong mỗi khối, ta đảo đúng phần đề yêu cầu: `min(k, phần còn lại)` ký tự đầu. Ba trường hợp của đề đều rơi vào công thức này nhờ slicing tự cắt cụt.
- Thứ tự các khối được giữ nguyên vì ta `append` theo `i` tăng dần.

⇒ Kết quả cuối đúng với mọi `s`, `k`.

## 11. Độ phức tạp

Gọi `n = len(s)`.

| Cách | Time | Space (thêm) | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng `if` đầy đủ | `O(n)` | `O(n)` | dài, dễ sai nhánh |
| 2. Slicing gọn | `O(n)` | `O(n)` | ngắn, dễ đọc — **khuyên dùng** |
| 3. Hai con trỏ trên list | `O(n)` | `O(n)` cho list (Python bắt buộc vì chuỗi bất biến) | giống C++/Java |
| 4. Ánh xạ chỉ số | `O(n)` | `O(n)` | luyện tư duy index |

Mỗi ký tự được đọc/ghi một số lần hằng số ⇒ `O(n)`. Không có cách nào dưới `O(n)` vì kết quả dài `n`.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu các cách

Nhóm test:
1. Mẫu đề.
2. `k = 1` → không đảo gì (đảo 1 ký tự = giữ nguyên).
3. `k >= n` → đảo toàn bộ.
4. `n` là bội của `2k`, `n` dư đúng `k`, dư ít hơn `k`.
5. Ngẫu nhiên: so sánh 4 cách với nhau.

In [18]:
tests = [
    ("abcdefg", 2, "bacdfeg"),
    ("abcd", 2, "bacd"),
    ("a", 1, "a"),
    ("abcdef", 1, "abcdef"),
    ("abc", 5, "cba"),
    ("abc", 3, "cba"),
    ("abcdefgh", 2, "bacdfegh"),     # n là bội của 2k
    ("abcdef", 2, "bacdfe"),         # dư đúng k
    ("abcde", 2, "bacde"),           # dư 1 < k
    ("abcdefghij", 3, "cbadefihgj"),
]
methods = [reverse_str_explicit, reverse_str_slice, reverse_str_two_pointers,
           reverse_str_slice_assign, reverse_str_index]
for s, k, expected in tests:
    for f in methods:
        got = f(s, k)
        assert got == expected, (f.__name__, s, k, got, expected)
    print(f"{s!r:>13}, k={k} -> {expected!r} ✔")
print("Tất cả test PASS!")

    'abcdefg', k=2 -> 'bacdfeg' ✔
       'abcd', k=2 -> 'bacd' ✔
          'a', k=1 -> 'a' ✔
     'abcdef', k=1 -> 'abcdef' ✔
        'abc', k=5 -> 'cba' ✔
        'abc', k=3 -> 'cba' ✔
   'abcdefgh', k=2 -> 'bacdfegh' ✔
     'abcdef', k=2 -> 'bacdfe' ✔
      'abcde', k=2 -> 'bacde' ✔
 'abcdefghij', k=3 -> 'cbadefihgj' ✔
Tất cả test PASS!


In [19]:
import random

random.seed(541)
for _ in range(500):
    n = random.randint(1, 30)
    s = "".join(random.choice("abcxyz") for _ in range(n))
    k = random.randint(1, 35)
    answers = {f(s, k) for f in methods}
    assert len(answers) == 1, (s, k, answers)
print("500 test ngẫu nhiên: 5 cách cho cùng kết quả.")

500 test ngẫu nhiên: 5 cách cho cùng kết quả.


## 13. Lỗi người mới hay gặp

### Lỗi 1: Nhảy bước `k` thay vì `2k`

In [20]:
def wrong_step_k(s: str, k: int) -> str:
    parts = []
    for i in range(0, len(s), k):          # SAI: bước k
        parts.append(s[i:i + k][::-1])
    return "".join(parts)

print(wrong_step_k("abcdefg", 2), "| đúng:", reverse_str_slice("abcdefg", 2))

badcfeg | đúng: bacdfeg


Vì sao sai: bước `k` khiến **mọi** đoạn đều bị đảo, kể cả nửa "giữ nguyên". Sửa: bước `2 * k` và nối thêm phần giữ `s[i+k:i+2k]`.

### Lỗi 2: Quên phần giữ nguyên

Viết `parts.append(s[i:i+k][::-1])` mà quên `parts.append(s[i+k:i+2*k])` → mất ký tự.

In [21]:
def wrong_forget_tail(s: str, k: int) -> str:
    parts = []
    for i in range(0, len(s), 2 * k):
        parts.append(s[i:i + k][::-1])       # thiếu phần giữ
    return "".join(parts)

print(repr(wrong_forget_tail("abcdefg", 2)), "| đúng:", repr(reverse_str_slice("abcdefg", 2)))

'bafe' | đúng: 'bacdfeg'


### Lỗi 3: Hai con trỏ không dùng `min` → `IndexError`

`right = i + k - 1` có thể vượt `n - 1` ở khối cuối. Phải viết `right = min(i + k - 1, n - 1)`.

In [22]:
def wrong_no_min(s: str, k: int) -> str:
    chars = list(s)
    for i in range(0, len(chars), 2 * k):
        left, right = i, i + k - 1           # SAI: có thể vượt biên
        while left < right:
            chars[left], chars[right] = chars[right], chars[left]
            left += 1
            right -= 1
    return "".join(chars)

try:
    print(wrong_no_min("abc", 5))
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


### Lỗi 4: Cố sửa chuỗi trực tiếp `s[i] = ...`

Chuỗi bất biến → `TypeError`. Sửa: dùng `list(s)` hoặc slicing tạo chuỗi mới (xem mục 2.3).

### Lỗi 5: Dùng `reversed(...)` mà quên `join`

`reversed("ab")` trả về **iterator**, không phải chuỗi. Nối `"x" + reversed("ab")` sẽ lỗi. Dùng `"".join(reversed(...))` hoặc `[::-1]`.

In [23]:
r = reversed("ab")
print(r)                      # <reversed object ...>
print("".join(reversed("ab")))  # ba

ba


## 14. Mẹo nhận dạng pattern và biến thể

### Mẹo nhận dạng

- Đề nói "**cứ mỗi** `m` phần tử làm gì đó" → nghĩ ngay `for i in range(0, n, m)` và xử lý khối `s[i:i+m]`.
- Đề có các trường hợp "**nếu còn ít hơn ...**" ở cuối → thử xem slicing tự cắt cụt có xử lý hộ không trước khi viết `if`.
- Đảo đoạn tại chỗ → **hai con trỏ** `left`, `right` tiến vào giữa.
- Cần biết "phần tử ở vị trí `j` đến từ đâu" → **ánh xạ chỉ số** bằng `%` và `//`.

### Biến thể

1. **LeetCode 344 – Reverse String**: đảo toàn bộ mảng ký tự tại chỗ (hai con trỏ cơ bản).
2. **LeetCode 557 – Reverse Words in a String III**: đảo từng từ (khối phân tách bởi dấu cách).
3. Đảo **mọi** khối `k` (không bỏ khối xen kẽ) — chính là `wrong_step_k` ở trên, nhưng là đúng cho đề khác.
4. **LeetCode 25 – Reverse Nodes in k-Group** (Hard): ý tưởng tương tự trên linked list.

In [24]:
# Biến thể 3 demo: đảo mọi khối k
def reverse_every_k(s: str, k: int) -> str:
    return "".join(s[i:i + k][::-1] for i in range(0, len(s), k))

print(reverse_every_k("abcdefg", 3))   # cba fed g -> cbafedg

cbafedg


Giải thích `"".join(... for ... in ...)`: đây là **generator expression** — giống list comprehension nhưng không tạo list trung gian; `join` nhận trực tiếp từng phần.

## 15. Bài tập tự luyện

### Bài 1 (dễ)
Viết `split_blocks(s, m)` trả về list các khối độ dài `m` (khối cuối có thể ngắn hơn).

```python
split_blocks("abcdefg", 3) -> ["abc", "def", "g"]
```

In [25]:
def split_blocks(s: str, m: int) -> list[str]:
    # TODO: dùng range(0, len(s), m) và slicing
    pass

# assert split_blocks("abcdefg", 3) == ["abc", "def", "g"]
# assert split_blocks("ab", 5) == ["ab"]

### Bài 2 (dễ)
Viết `reverse_range(s, a, b)` trả về chuỗi mới với đoạn `s[a..b]` (bao gồm cả `b`) bị đảo.

```python
reverse_range("abcdef", 1, 3) -> "adcbef"
```

In [26]:
def reverse_range(s: str, a: int, b: int) -> str:
    # TODO
    pass

# assert reverse_range("abcdef", 1, 3) == "adcbef"
# assert reverse_range("abc", 0, 2) == "cba"

### Bài 3 (vừa)
Giải lại bài 541 **bằng hai con trỏ**, không dùng `[::-1]` và không dùng `reversed`.

In [27]:
def reverse_str_ex(s: str, k: int) -> str:
    # TODO: list(s) + while left < right + min(...)
    pass

# assert reverse_str_ex("abcdefg", 2) == "bacdfeg"
# assert reverse_str_ex("abc", 5) == "cba"

### Bài 4 (vừa)
Biến thể: cứ mỗi `2k` ký tự, **giữ** `k` ký tự đầu và **đảo** `k` ký tự sau.

```python
reverse_second_half("abcdefg", 2) -> "abdcefg"
```

In [28]:
def reverse_second_half(s: str, k: int) -> str:
    # TODO
    pass

# assert reverse_second_half("abcdefg", 2) == "abdcefg"
# assert reverse_second_half("abcdefgh", 2) == "abdcefhg"

### Bài 5 (khó hơn)
Viết `reverse_str_inverse(t, k)`: cho kết quả `t` của bài 541, tìm lại chuỗi gốc `s`. (Gợi ý: phép biến đổi này có gì đặc biệt khi áp dụng hai lần?)

In [29]:
def reverse_str_inverse(t: str, k: int) -> str:
    # TODO
    pass

# assert reverse_str_inverse("bacdfeg", 2) == "abcdefg"

## 16. Gợi ý 3 mức

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Bước nhảy là `m` | `s[i:i+m]` | list comprehension `[s[i:i+m] for i in range(0, len(s), m)]` |
| 2 | Ba phần: trước, giữa, sau | `s[:a]`, `s[a:b+1]`, `s[b+1:]` | Đảo phần giữa bằng `[::-1]` |
| 3 | Đổi sang list | `right = min(i+k-1, n-1)` | swap rồi `left += 1; right -= 1` |
| 4 | Vẫn bước `2k` | Phần giữ là `s[i:i+k]` | Phần đảo là `s[i+k:i+2k][::-1]` |
| 5 | Đảo một đoạn hai lần thì sao? | Các khối không đổi vị trí | Áp dụng lại chính bài 541 |

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [30]:
def answer_split_blocks(s: str, m: int) -> list[str]:
    return [s[i:i + m] for i in range(0, len(s), m)]

def answer_reverse_range(s: str, a: int, b: int) -> str:
    return s[:a] + s[a:b + 1][::-1] + s[b + 1:]

def answer_reverse_str_ex(s: str, k: int) -> str:
    chars = list(s)
    n = len(chars)
    for i in range(0, n, 2 * k):
        left, right = i, min(i + k - 1, n - 1)
        while left < right:
            chars[left], chars[right] = chars[right], chars[left]
            left += 1
            right -= 1
    return "".join(chars)

def answer_reverse_second_half(s: str, k: int) -> str:
    parts = []
    for i in range(0, len(s), 2 * k):
        parts.append(s[i:i + k])
        parts.append(s[i + k:i + 2 * k][::-1])
    return "".join(parts)

def answer_reverse_str_inverse(t: str, k: int) -> str:
    # Phép biến đổi là "tự nghịch đảo": áp dụng 2 lần trả lại chuỗi gốc
    return reverse_str_slice(t, k)

assert answer_split_blocks("abcdefg", 3) == ["abc", "def", "g"]
assert answer_split_blocks("ab", 5) == ["ab"]
assert answer_reverse_range("abcdef", 1, 3) == "adcbef"
assert answer_reverse_range("abc", 0, 2) == "cba"
assert answer_reverse_str_ex("abcdefg", 2) == "bacdfeg"
assert answer_reverse_str_ex("abc", 5) == "cba"
assert answer_reverse_second_half("abcdefg", 2) == "abdcefg"
assert answer_reverse_second_half("abcdefgh", 2) == "abdcefhg"
assert answer_reverse_str_inverse("bacdfeg", 2) == "abcdefg"
for _ in range(200):
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 20)))
    k = random.randint(1, 8)
    assert answer_reverse_str_inverse(reverse_str_slice(s, k), k) == s
print("Đáp án chạy PASS toàn bộ test!")

Đáp án chạy PASS toàn bộ test!


## 18. Template nhớ nhanh và checklist

### Template

```python
class Solution:
    def reverseStr(self, s: str, k: int) -> str:
        parts = []
        for i in range(0, len(s), 2 * k):
            parts.append(s[i:i + k][::-1])   # đảo k đầu (tự cắt cụt nếu thiếu)
            parts.append(s[i + k:i + 2 * k]) # giữ k sau
        return "".join(parts)
```

### Checklist trước khi submit

- [ ] Bước nhảy là `2 * k`, không phải `k`.
- [ ] Có nối phần giữ nguyên `s[i+k:i+2k]`.
- [ ] Nếu dùng hai con trỏ: `right = min(i + k - 1, n - 1)`.
- [ ] Không gán trực tiếp vào chuỗi; dùng list hoặc slicing.
- [ ] Đã thử `k = 1`, `k >= len(s)`, độ dài dư đúng `k` và dư < `k`.
- [ ] Trả về **chuỗi** (`"".join(...)`), không trả list.